In [2]:
from transformers import GPT2LMHeadModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

torch.set_default_device(device='cuda')
torch.manual_seed(1234)
torch.cuda.manual_seed(1234)

In [3]:
import tiktoken

In [4]:
class DataLoader:
    def __init__(self, batch_size, block_size, input_file):
        self.pos = 0
        self.epoch = 0
        self.batch_size = batch_size
        self.block_size = block_size
        with open(input_file, 'r') as f: self.data = f.read()
        encoder = tiktoken.get_encoding('gpt2')
        self.data = torch.tensor(encoder.encode(self.data))
    def __call__(self):
        buffer = self.data[self.pos:self.pos+(self.block_size*self.batch_size)+1]
        x = buffer[:-1].view(self.batch_size, -1)
        y = buffer[1:].view(self.batch_size, -1)
        self.pos += self.block_size * self.batch_size
        if self.pos >= self.data.shape[0]:
            self.pos = 0
            print(f"epoch {self.epoch} tamamlandi")
            self.epoch+=1
        return x,y

In [5]:
import urllib
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

('shakespeare.txt', <http.client.HTTPMessage at 0x7d9e8a5bccd0>)

In [11]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.residual = True
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))
    def forward(self,x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2) 
        # n_embd x (3 * n_embd) boyutlu tek c_attn katmanini uce ayirir. boylece q,k,v matrislerinin hepsi ayri bir girdiymis gibi, batch icindeki elemanlar gibi hesaplanir.
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1,2) 
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        # q,k,v'nin icindeki headler ayrilir, headler birbirlerinden izole edilerek batch dimension haline getilir. dort boyutlu tensorlar olusmus olur
        # transpoze edilmesinin sebebi de her headin ayri bir batch dimension olarak alinmasi, T yani ornekler boyutu ile C yani kanallar her head icinde bagimsiz isleme alinmalidir.
        attn = q @ k.transpose(-2,-1)
        attn = attn * torch.sqrt(torch.tensor(self.n_embd//self.n_head))**-1
        attn = attn.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = attn @ v
        out = out.transpose(1,2).contiguous().view(B, T, C)
        # hesaplamalar yapildiktan sonra tekrar eski haline getiriliyor
        out = self.c_proj(out)
        return out

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.residual = True
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config=config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config=config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size:int = 1024
    vocab_size:int = 50257
    n_layers:int = 12
    n_head:int = 12
    n_embd:int = 768
    weight_tying:bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(wte = nn.Embedding(config.vocab_size, config.n_embd), wpe = nn.Embedding(config.block_size, config.n_embd), h = nn.ModuleList([Block(config) for _ in range(config.n_layers)]), ln_f = nn.LayerNorm(config.n_embd)))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        if config.weight_tying: self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02 if not hasattr(module, 'residual') else 0.02*(2*self.config.n_layers)**-0.5)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    @classmethod
    def from_pretrained(cls, model_type):
        configs = {
            'gpt2': dict(n_layers = 12, n_head = 12, n_embd = 768),
            'gpt2-medium': dict(n_layers = 24, n_head = 16, n_embd = 1024),
            'gpt2-large': dict(n_layers = 36, n_head = 20, n_embd = 1280), 'gpt2-xl': dict(n_layers = 48, n_head = 25, n_embd = 1600)
        }[model_type]
        configs['vocab_size'] = 50257
        configs['block_size'] = 1024
        config = GPTConfig(**configs)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = [k for k in sd.keys() if not k.endswith('.attn.masked_bias')]

        hf_model = GPT2LMHeadModel.from_pretrained(model_type)
        hf_sd = hf_model.state_dict()
        hf_sd_keys = [k for k in hf_sd.keys() if not k.endswith('.attn.masked_bias') and not k.endswith('.attn.bias') and not (k.endswith('lm_head.weight') if config.weight_tying else False)]
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

        for k in hf_sd_keys:
            if any(k.endswith(w) for w in transposed):
                assert hf_sd[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(hf_sd[k].t())
            else:
                assert hf_sd[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(hf_sd[k])
        return model
    def forward(self, x, targets=None):
        token_embeddings = self.transformer.wte(x)
        positional_embeddings = self.transformer.wpe(torch.arange(x.shape[1],device='cuda'))
        x = token_embeddings + positional_embeddings
        for block in self.transformer['h']:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        if targets==None:
            return logits
        else:
            return (F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1)), logits) # (loss, logits) dondurulur
            # batch dimensionlari duzlestirilerek karsilastirma islemleri paralellestirilir

In [ ]:
import time


config = GPTConfig()
model = GPT(config)
model = model.to('cuda')
model = torch.compile(model)
optim = torch.optim.AdamW(model.parameters(),lr=5e-4)
data_loader = DataLoader(4,1024,"shakespeare.txt")


for i in range(500):
    beginning_time = time.time()
    x,y = data_loader()
    x,y = x.to('cuda'), y.to('cuda')
    optim.zero_grad()
    loss, logits = model(x,y)
    loss.backward()
    optim.step()
    #torch.cuda.synchronize()
    #end_time = time.time()
    #time_spent = (end_time-beginning_time)*1000
    #tps = (data_loader.block_size*data_loader.batch_size/time_spent)*1000
    print(f"loss {loss.item()}")

loss 10.936405181884766
loss 9.594493865966797
loss 9.399567604064941
loss 9.013469696044922
loss 8.886816024780273
loss 8.508896827697754
loss 8.364521026611328
loss 8.116840362548828
loss 8.012655258178711
loss 7.810754776000977
loss 7.8486127853393555
loss 7.807686805725098
loss 7.856607437133789
loss 7.7232818603515625


KeyboardInterrupt: 

In [ ]:
param_count = sum(p.numel() for p in model.parameters())
param_count

124439808

In [ ]:
import time


config = GPTConfig()
model = GPT(config)
model = model.to('cuda')
model = torch.compile(model)
optim = torch.optim.AdamW(model.parameters(),lr=5e-4)
data_loader = DataLoader(4,1024,"shakespeare.txt")


for i in range(500):
    beginning_time = time.time()
    x,y = data_loader()
    x,y = x.to('cuda'), y.to('cuda')
    with torch.autocast('cuda', torch.float16):
        loss, logits = model(x,y)
    optim.zero_grad()
    loss.backward()
    optim.step()
    torch.cuda.synchronize()
    end_time = time.time()
    time_spent = (end_time-beginning_time)*1000
    tps = (data_loader.block_size*data_loader.batch_size/time_spent)*1000
    print(f"{tps:.1f} token/s - {time_spent:.3f}/step - loss {loss.item()}")

124439808

In [28]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.residual = True
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))
    def forward(self,x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2) 
        # n_embd x (3 * n_embd) boyutlu tek c_attn katmanini uce ayirir. boylece q,k,v matrislerinin hepsi ayri bir girdiymis gibi, batch icindeki elemanlar gibi hesaplanir.
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1,2) 
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        

        # q,k,v'nin icindeki headler ayrilir, headler birbirlerinden izole edilerek batch dimension haline getilir. dort boyutlu tensorlar olusmus olur
        # transpoze edilmesinin sebebi de her headin ayri bir batch dimension olarak alinmasi, T yani ornekler boyutu ile C yani kanallar her head icinde bagimsiz isleme alinmalidir.
        """        attn = q @ k.transpose(-2,-1)
        attn = attn * torch.sqrt(torch.tensor(self.n_embd//self.n_head))**-1
        attn = attn.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)"""
        out = F.scaled_dot_product_attention(q,k,v, is_causal=True)
        
        out = out.transpose(1,2).contiguous().view(B, T, C)
        # hesaplamalar yapildiktan sonra tekrar eski haline getiriliyor
        out = self.c_proj(out)
        return out

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.residual = True
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config=config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config=config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size:int = 1024
    vocab_size:int = 50257
    n_layers:int = 12
    n_head:int = 12
    n_embd:int = 768
    weight_tying:bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(wte = nn.Embedding(config.vocab_size, config.n_embd), wpe = nn.Embedding(config.block_size, config.n_embd), h = nn.ModuleList([Block(config) for _ in range(config.n_layers)]), ln_f = nn.LayerNorm(config.n_embd)))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        if config.weight_tying: self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02 if not hasattr(module, 'residual') else 0.02*(2*self.config.n_layers)**-0.5)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    @classmethod
    def from_pretrained(cls, model_type):
        configs = {
            'gpt2': dict(n_layers = 12, n_head = 12, n_embd = 768),
            'gpt2-medium': dict(n_layers = 24, n_head = 16, n_embd = 1024),
            'gpt2-large': dict(n_layers = 36, n_head = 20, n_embd = 1280), 'gpt2-xl': dict(n_layers = 48, n_head = 25, n_embd = 1600)
        }[model_type]
        configs['vocab_size'] = 50257
        configs['block_size'] = 1024
        config = GPTConfig(**configs)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = [k for k in sd.keys() if not k.endswith('.attn.masked_bias')]

        hf_model = GPT2LMHeadModel.from_pretrained(model_type)
        hf_sd = hf_model.state_dict()
        hf_sd_keys = [k for k in hf_sd.keys() if not k.endswith('.attn.masked_bias') and not k.endswith('.attn.bias') and not (k.endswith('lm_head.weight') if config.weight_tying else False)]
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

        for k in hf_sd_keys:
            if any(k.endswith(w) for w in transposed):
                assert hf_sd[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(hf_sd[k].t())
            else:
                assert hf_sd[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(hf_sd[k])
        return model
    def forward(self, x, targets=None):
        token_embeddings = self.transformer.wte(x)
        positional_embeddings = self.transformer.wpe(torch.arange(x.shape[1],device='cuda'))
        x = token_embeddings + positional_embeddings
        for block in self.transformer['h']:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        if targets==None:
            return logits
        else:
            return (F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1)), logits) # (loss, logits) dondurulur
            # batch dimensionlari duzlestirilerek karsilastirma islemleri paralellestirilir

In [29]:
import time


config = GPTConfig()
model = GPT(config)
model = model.to('cuda')
model = torch.compile(model)
optim = torch.optim.AdamW(model.parameters(),lr=5e-4)
data_loader = DataLoader(4,1024,"shakespeare.txt")


for i in range(500):
    beginning_time = time.time()
    x,y = data_loader()
    x,y = x.to('cuda'), y.to('cuda')
    loss, logits = model(x,y)
    optim.zero_grad()
    loss.backward()
    optim.step()
    torch.cuda.synchronize()
    end_time = time.time()
    time_spent = (end_time-beginning_time)*1000
    tps = (data_loader.block_size*data_loader.batch_size/time_spent)*1000
    print(f"{tps:.1f} token/s - {time_spent:.3f}/step - loss {loss.item()}")

204.1 token/s - 20067.688/step - loss 10.975517272949219
3602.2 token/s - 1137.070/step - loss 9.581748008728027
3515.0 token/s - 1165.277/step - loss 9.147086143493652
3542.1 token/s - 1156.377/step - loss 8.72867202758789
3501.8 token/s - 1169.686/step - loss 8.350564956665039
3494.7 token/s - 1172.076/step - loss 7.832098007202148
3470.1 token/s - 1180.381/step - loss 7.495909690856934
3421.8 token/s - 1197.018/step - loss 7.157910346984863
3383.5 token/s - 1210.566/step - loss 7.03164529800415
3372.9 token/s - 1214.402/step - loss 6.7724199295043945
3325.8 token/s - 1231.580/step - loss 6.826852798461914
3311.0 token/s - 1237.105/step - loss 6.928559303283691
3273.4 token/s - 1251.284/step - loss 7.156374931335449
3211.9 token/s - 1275.266/step - loss 7.1307244300842285
3182.7 token/s - 1286.975/step - loss 6.832464694976807
3126.6 token/s - 1310.047/step - loss 6.89678955078125
3103.4 token/s - 1319.846/step - loss 6.837932109832764
3061.1 token/s - 1338.081/step - loss 6.64422988

KeyboardInterrupt: 

In [30]:
import time


config = GPTConfig(vocab_size=50304)
model = GPT(config)
model = model.to('cuda')
model = torch.compile(model)
optim = torch.optim.AdamW(model.parameters(),lr=5e-4)
data_loader = DataLoader(4,1024,"shakespeare.txt")


for i in range(500):
    beginning_time = time.time()
    x,y = data_loader()
    x,y = x.to('cuda'), y.to('cuda')
    loss, logits = model(x,y)
    optim.zero_grad()
    loss.backward()
    optim.step()
    torch.cuda.synchronize()
    end_time = time.time()
    time_spent = (end_time-beginning_time)*1000
    tps = (data_loader.block_size*data_loader.batch_size/time_spent)*1000
    print(f"{tps:.1f} token/s - {time_spent:.3f}/step - loss {loss.item()}")

128.0 token/s - 32009.591/step - loss 11.0377779006958
3037.2 token/s - 1348.611/step - loss 9.570230484008789
2978.8 token/s - 1375.058/step - loss 9.272995948791504
2921.6 token/s - 1401.952/step - loss 8.6882905960083
2890.7 token/s - 1416.975/step - loss 8.371349334716797
2602.3 token/s - 1573.991/step - loss 7.898865699768066
2138.8 token/s - 1915.083/step - loss 7.566180229187012
1831.7 token/s - 2236.137/step - loss 7.216507434844971
1689.1 token/s - 2425.000/step - loss 7.054150581359863
2412.5 token/s - 1697.834/step - loss 6.79246711730957
2446.9 token/s - 1673.939/step - loss 6.849671840667725
2677.4 token/s - 1529.824/step - loss 6.956175327301025
2912.8 token/s - 1406.184/step - loss 7.144424915313721
2957.7 token/s - 1384.866/step - loss 7.125770092010498
2986.8 token/s - 1371.351/step - loss 6.822954177856445
3043.3 token/s - 1345.900/step - loss 6.891253471374512
3068.3 token/s - 1334.945/step - loss 6.833667278289795
3096.2 token/s - 1322.927/step - loss 6.665339469909

KeyboardInterrupt: 